In [1]:
import sys, os
from pathlib import Path

ROOT = Path(r"C:\Users\user\doc-asistant\doc-asistant")
sys.path.append(str(ROOT))
sys.path.append(str(ROOT / "notebooks" / "from_src"))

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("vram free/total:", torch.cuda.mem_get_info())
print("cwd:", os.getcwd())

torch: 2.6.0+cu124
cuda: True
gpu: NVIDIA GeForce RTX 3060
vram free/total: (11799625728, 12884377600)
cwd: C:\Users\user\doc-asistant\doc-asistant\notebooks


In [2]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

import config
from pathlib import Path

# патч путей (как в 01 ноутбуке)
ROOT = Path(r"C:\Users\user\doc-asistant\doc-asistant")
config.ROOT = ROOT
config.LORA_DIR = ROOT / "models" / "lora-doc-assistant"
config.LORA_DIR.mkdir(parents=True, exist_ok=True)

print("LORA_DIR:", config.LORA_DIR)
print("HF_TOKEN:", "OK" if config.HF_TOKEN and config.HF_TOKEN.startswith("hf_") else "MISSING")

LORA_DIR: C:\Users\user\doc-asistant\doc-asistant\models\lora-doc-assistant
HF_TOKEN: OK


In [3]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=config.HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=config.HF_TOKEN,
)

model = prepare_model_for_kbit_training(model)

# фикс: каст bf16-параметров в fp32 (если они есть)
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

print("model loaded")
print("dtype:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model loaded
dtype: torch.float32


In [4]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [5]:
import json
from pathlib import Path

DATASET_PATH = ROOT / "data" / "qa_dataset.json"
print("exists:", DATASET_PATH.exists())
print("size:", DATASET_PATH.stat().st_size, "bytes")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

print("type:", type(raw).__name__)
print("entries:", len(raw))
print()
print("first entry (raw):")
print(repr(raw[0])[:600])
print()
print("keys of first entry:", list(raw[0].keys()) if isinstance(raw[0], dict) else "not a dict")

exists: True
size: 5530 bytes
type: list
entries: 20

first entry (raw):
{'text': "<|system|>\nТы аналитик. Отвечай кратко, только числами. Если ответа нет в данных - пиши 'Нет данных'.\n<|user|>\nСколько клиентов в сегменте А?\n<|assistant|>\n42"}

keys of first entry: ['text']


In [9]:
import sys
import json
import pandas as pd
from pathlib import Path

# --- data_loader из from_src ---
sys.path.append(str(ROOT / "notebooks" / "from_src"))
import data_loader

# --- загрузка рецептов ---
CSV = ROOT / "data" / "cooking_clean.csv"
df = pd.read_csv(CSV, sep=";")
print("total recipes:", len(df))
print("columns:", df.columns.tolist())

# --- категория по title ---
df["category"] = df["title"].apply(data_loader.get_category)
print("categories:", df["category"].value_counts().to_dict())

# --- фильтр: dessert + bread ---
baking = df[df["category"].isin(["dessert", "bread"])].copy()
print("dessert + bread:", len(baking))

# --- генерация Q&A ---
SYSTEM = (
    "Ты помощник по выпечке. Отвечай кратко, только фактами из рецепта. "
    "Если ответа нет в данных - пиши 'Нет данных'."
)

dataset = []

# 1) Расскажи про рецепт
for _, row in baking.head(15).iterrows():
    title = str(row["title"]).strip()
    instr = str(row["instructions"]).strip().replace("\n", " ")[:400]
    text = (
        f"<|system|>\n{SYSTEM}\n"
        f"<|user|>\nРасскажи кратко про рецепт: {title}\n"
        f"<|assistant|>\n{instr}"
    )
    dataset.append({"text": text})

# 2) Ингредиенты
for _, row in baking.head(10).iterrows():
    title = str(row["title"]).strip()
    instr = str(row["instructions"]).strip().replace("\n", " ")[:200]
    text = (
        f"<|system|>\n{SYSTEM}\n"
        f"<|user|>\nКакие ингредиенты в рецепте '{title}'?\n"
        f"<|assistant|>\n{instr}"
    )
    dataset.append({"text": text})

# 3) Нет данных
for q in [
    "Сколько сахара в рецепте 'Пицца Маргарита'?",
    "Расскажи про рецепт 'Борщ с пампушками'",
    "Какие ингредиенты в 'Суши Филадельфия'?",
    "Сколько калорий в 'Шоколадном торте'?",
    "Какой рецепт лучше для похудения?",
]:
    text = (
        f"<|system|>\n{SYSTEM}\n"
        f"<|user|>\n{q}\n"
        f"<|assistant|>\nНет данных"
    )
    dataset.append({"text": text})

# --- сохранение ---
OUT = ROOT / "data" / "qa_dataset.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print("saved:", OUT)
print("entries:", len(dataset))
print()
print("=== example 1 ===")
print(dataset[0]["text"][:400])
print()
print("=== example no data ===")
print(dataset[-1]["text"])

total recipes: 500
columns: ['number', 'title', 'instructions']
categories: {'other': 184, 'dessert': 155, 'meat': 79, 'bread': 58, 'soup': 14, 'salad': 10}
dessert + bread: 213
saved: C:\Users\user\doc-asistant\doc-asistant\data\qa_dataset.json
entries: 30

=== example 1 ===
<|system|>
Ты помощник по выпечке. Отвечай кратко, только фактами из рецепта. Если ответа нет в данных - пиши 'Нет данных'.
<|user|>
Расскажи кратко про рецепт: peanut butter cup cookies
<|assistant|>
1/2 c butter or margarine 1/2 c peanut butter 1/2 c brown sugar 1/2 c white sugar 1 egg 1/2 tsp vanilla 1 1/4 c sifted flour 3/4 tsp baking soda 1/2 tsp salt 1 10 oz bag miniature reeses peanut butte

=== example no data ===
<|system|>
Ты помощник по выпечке. Отвечай кратко, только фактами из рецепта. Если ответа нет в данных - пиши 'Нет данных'.
<|user|>
Какой рецепт лучше для похудения?
<|assistant|>
Нет данных


In [10]:
import json
with open(ROOT / "data" / "qa_dataset.json", "r", encoding="utf-8") as f:
    check = json.load(f)

print("entries:", len(check))
print("all have 'text':       ", all("text" in e for e in check))
print("all have <|system|>:   ", all("<|system|>" in e["text"] for e in check))
print("all have <|user|>:     ", all("<|user|>" in e["text"] for e in check))
print("all have <|assistant|>:", all("<|assistant|>" in e["text"] for e in check))
print("no_data count:         ", sum(1 for e in check if e["text"].rstrip().endswith("Нет данных")))

print()
print("--- user questions ---")
for i, e in enumerate(check):
    user_part = e["text"].split("<|user|>\n")[1].split("\n<|assistant|>")[0]
    print(f"{i+1:2d}. {user_part[:80]}")

entries: 30
all have 'text':        True
all have <|system|>:    True
all have <|user|>:      True
all have <|assistant|>: True
no_data count:          5

--- user questions ---
 1. Расскажи кратко про рецепт: peanut butter cup cookies
 2. Расскажи кратко про рецепт: colonial cheese biscuits
 3. Расскажи кратко про рецепт: apple sour cream coffee cake
 4. Расскажи кратко про рецепт: school girl picklessweet spear
 5. Расскажи кратко про рецепт: wedding cookies
 6. Расскажи кратко про рецепт: scottish shortbread
 7. Расскажи кратко про рецепт: chocolate eclair dessert
 8. Расскажи кратко про рецепт: banana nut muffins
 9. Расскажи кратко про рецепт: apple cream coffee cake
10. Расскажи кратко про рецепт: sour cream apple pie
11. Расскажи кратко про рецепт: yellow cakedelicious with just a dusting of powdered
12. Расскажи кратко про рецепт: rhubarb cake
13. Расскажи кратко про рецепт: onerise monkey bread
14. Расскажи кратко про рецепт: grans toll house cookies
15. Расскажи кратко про ре

In [12]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# --- датасет ---
ds = load_dataset(
    "json",
    data_files=str(ROOT / "data" / "qa_dataset.json"),
    split="train",
)
print("dataset:", ds)

# --- SFT конфиг (bf16 вместо fp16) ---
sft_config = SFTConfig(
    output_dir=str(config.LORA_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    max_length=256,
    bf16=True,                 # ← было fp16=True
    optim="adamw_torch",       # ← уже стоит
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=sft_config,
    processing_class=tokenizer,
)

train_result = trainer.train()
print("done")
print(train_result)

dataset: Dataset({
    features: ['text'],
    num_rows: 30
})


Step,Training Loss
1,2.873688
2,2.732493
3,2.834305
4,2.664070
5,2.563647
6,2.520388
7,2.354258
8,2.190517
9,2.398776
10,2.056574


done
TrainOutput(global_step=12, training_loss=2.4561578234036765, metrics={'train_runtime': 63.4445, 'train_samples_per_second': 1.419, 'train_steps_per_second': 0.189, 'total_flos': 103195547320320.0, 'train_loss': 2.4561578234036765, 'epoch': 3.0})


In [13]:
# --- сохранение адаптера ---
model.save_pretrained(str(config.LORA_DIR))
tokenizer.save_pretrained(str(config.LORA_DIR))

print("saved to:", config.LORA_DIR)
print()

import os
for f in sorted(os.listdir(config.LORA_DIR)):
    full = config.LORA_DIR / f
    if full.is_file():
        print(f"  {f:40s}  {full.stat().st_size:>10,} bytes")
    else:
        print(f"  {f:40s}  <DIR>")

saved to: C:\Users\user\doc-asistant\doc-asistant\models\lora-doc-assistant

  README.md                                      1,504 bytes
  adapter_config.json                            1,179 bytes
  adapter_model.safetensors                  4,387,728 bytes
  chat_template.jinja                            2,561 bytes
  checkpoint-12                             <DIR>
  tokenizer.json                            11,421,892 bytes
  tokenizer_config.json                            724 bytes


In [15]:
SYSTEM = (
    "Ты помощник по выпечке. Отвечай кратко, только фактами из рецепта. "
    "Если ответа нет в данных - пиши 'Нет данных'."
)

def generate(model, tokenizer, question, max_new_tokens=100):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()


test_questions = [
    "Расскажи кратко про рецепт: chocolate eclair dessert",
    "Какие ингредиенты в рецепте 'wedding cookies'?",
    "Расскажи про рецепт 'Борщ с пампушками'",
    "Сколько калорий в 'Шоколадном торте'?",
    "Какой рецепт лучше для похудения?",
]

print("=" * 80)
print("BASELINE (Qwen2.5 без LoRA)")
print("=" * 80)
model.eval()
with model.disable_adapter():
    for i, q in enumerate(test_questions, 1):
        print(f"\n[{i}] {q}")
        print(generate(model, tokenizer, q, max_new_tokens=80))

print("\n\n" + "=" * 80)
print("FINE-TUNED (Qwen2.5 + LoRA)")
print("=" * 80)
model.eval()
for i, q in enumerate(test_questions, 1):
    print(f"\n[{i}] {q}")
    print(generate(model, tokenizer, q, max_new_tokens=80))

BASELINE (Qwen2.5 без LoRA)

[1] Расскажи кратко про рецепт: chocolate eclair dessert
Этот десерт представляет собой слоеный шоколадный пирог с конфитюром и мороженым. Шоколадная эллипсовая форма с конфитюрой на верху, затем мороженое покрыто шоколадной глазурью.

[2] Какие ингредиенты в рецепте 'wedding cookies'?
В рецепте "Wedding Cookies" обычно используются следующие ингредиенты:

- Яйца
- Сахарная пудра
- Мука
- Ванильный сахар
- Крем для торта (обычно используется молочный)
- Гашенфельд или другие сухие дрожжи
- Ц

[3] Расскажи про рецепт 'Борщ с пампушками'
Извините за путаницу, но я не могу помочь вам с выпечкой или борщем с пампушками. Моя задача — помогать в вопросах и проблемах, связанных с питанием и здоровьем. Возможно, вы могли бы попросить совета о том, как приготовить борщ

[4] Сколько калорий в 'Шоколадном торте'?
100 г шоколадного торта содержат примерно 350 ккал.

[5] Какой рецепт лучше для похудения?
Рецепт "Пицца-пасту" с творогом и оливковым маслом.


FINE-TUNED (

## Сравнение: Qwen2.5-1.5B-Instruct vs Qwen2.5 + LoRA

Обучено: 30 примеров, 3 эпохи, LoRA r=8, target=q/k/v/o_proj.
Baseline — та же модель с `disable_adapter()`.

### Контрольные вопросы

| # | Вопрос | Baseline | LoRA |
| 1 | chocolate eclair dessert | выдумка (слоёный пирог) | ингредиенты из датасета + |
| 2 | wedding cookies | частичная выдумка | не рецепт, этикетка |
| 3 | Борщ с пампушками | отказ, не то | **Нет данных +** |
| 4 | калории в торте | 350 ккал (выдумка) | 1050 ккал (выдумка) |
| 5 | рецепт для похудения | пицца (выдумка) | общий ответ (выдумка) |

### Итог: 1 из 5

LoRA научила модель писать «Нет данных» на нерелевантный вопрос
(Борщ с пампушками), но не обобщила паттерн на остальные 4 случая.

### Почему слабо
- 30 примеров — слишком мало
- только 5 примеров «Нет данных»
- смешанные языки (рецепты EN, вопросы RU)
- 3 эпохи

### Что это значит
Для устойчивого эффекта нужно 500+ примеров, чистый RU-датасет,
5–10 эпох. Это мотивирует **RAG**: честность через контекст,
а не через обучение на маленькой выборке.